## Neural Network Encoder for content based recommendations

I implemented a classic logistic regression for recommending videos based on their content features. Now, I would like to use a more sophisticated model to compare the performance of both models in order to choose the best suitable one for my needs. 

This notebook aims to implement a neural network using Tensorflow to encode users and videos vectors and then compute the dot product to predict if a given user for a given video may like the it or not (actually we will try to aproximate the watch ratio that tells if the user like the video if the ratio is above 0.8).

The model use the following input vectors:

- a user vector with 18 features represented by the average rating of a given user for each video tag (or video category)
- an it vector with 32 features that holds the one hot encoding of its tags as well as the duration of the given video

The model will then pass the vectors through idependant dense layers that will "encode" separately both user and item vectors. Then those encodings will feed the next dot product that compute the watch ratio of the given user for the given video. The model follows the `NNEncoding` notebook in the course for the architecture of the neural network except for the number of hidden units and some other hyperparameters that differ to maximize how the model performs.


## Import librairies

In [1]:
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pandas as pd
import sys
sys.path.append('../')
import metrics

MANUAL_RANDOM_SEED=42
EMBEDDINGS_PATH='../embeddings'

## Load embeddings

In [ ]:
videos_content: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_content.pkl')
videos_tags: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_one_hot_tags.pkl')
user_avg_ratings: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_avg_ratings.pkl')
user_ratings: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_ratings.pkl')

# Keep only the video_duration field because we do not use any other information such
# as the views or the other popularity features
videos_content_full = videos_content
videos_content = videos_content[['video_duration']]
# Clip the watch ratio to help the model performs better
user_ratings['watch_ratio'] = user_ratings['watch_ratio'].apply(lambda x: 10 if x > 10 else x)

In [3]:
# Check the shapes for visualization!
videos_content.shape, videos_tags.shape, user_avg_ratings.shape, user_ratings.shape

((9937, 1), (9937, 31), (7176, 31), (10064065, 4))

## Preprocess the data

Now we can build the input vectors that is building both user and video vectors as describe in the introduction of the notebook. `video_content` is the matrix holding the video vectors whereas `user_content` is the one for the users.

In [4]:
# Build video vectors

# Merge video duration with the one hot encoding tags
videos_content = videos_content.merge(videos_tags, on='video_id')
# Now take all the ratings in the train set and concatenating the 
# video contents
videos_content = videos_content.merge(user_ratings, on='video_id')
# Sort values by video and user to match with the user vectors and
# prediction labels
videos_content = videos_content.sort_values(['video_id', 'user_id'])
videos_content = videos_content.reset_index(drop=True)
# Drop columns that we do not need for feeding the network
videos_content = videos_content.drop(['video_id', 'user_id', 'watch_ratio', 'like'], axis=1)

In [5]:
videos_content.head()

,video_duration,tag_0,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,...,tag_21,tag_22,tag_23,tag_24,tag_25,tag_26,tag_27,tag_28,tag_29,tag_30
0,5968,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,5968,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,5968,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,5968,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,5968,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# Build user vectors

# Merge ratings with the user content vectors, that is the average ratins of each user
user_content: pd.DataFrame = user_avg_ratings.merge(user_ratings, on='user_id')
# Sort to match with the other embeddings
user_content = user_content.sort_values(['video_id', 'user_id'])
user_content = user_content.reset_index(drop=True)
# Drop columns we do not need for feeding the model
user_content = user_content.drop(['video_id', 'user_id', 'watch_ratio', 'like'], axis=1)

In [7]:
user_content.head()

,avg_rating_tag_0,avg_rating_tag_1,avg_rating_tag_2,avg_rating_tag_3,avg_rating_tag_4,avg_rating_tag_5,avg_rating_tag_6,avg_rating_tag_7,avg_rating_tag_8,avg_rating_tag_9,...,avg_rating_tag_21,avg_rating_tag_22,avg_rating_tag_23,avg_rating_tag_24,avg_rating_tag_25,avg_rating_tag_26,avg_rating_tag_27,avg_rating_tag_28,avg_rating_tag_29,avg_rating_tag_30
0,0.428571,0.524590,0.363636,0.500000,0.307692,0.357724,0.262162,0.468354,0.519886,0.366071,...,0.166667,0.0,0.0,0.500000,0.360360,0.432653,0.333333,0.513684,0.666667,0.125000
1,1.000000,0.541667,0.500000,0.000000,0.666667,0.680000,0.472222,0.761905,0.744681,0.227273,...,0.000000,0.0,0.0,0.333333,0.473684,0.444444,1.000000,0.480519,0.000000,0.333333
2,0.444444,0.381679,0.347826,0.000000,0.375000,0.488000,0.363333,0.620915,0.503125,0.441176,...,0.571429,0.0,0.5,0.000000,0.524590,0.396985,0.250000,0.569620,0.000000,0.250000
3,0.800000,0.582278,0.416667,0.571429,0.642857,0.623188,0.436842,0.652542,0.540441,0.482759,...,0.000000,0.0,0.0,0.000000,0.520000,0.545455,0.500000,0.646707,0.000000,0.000000
4,0.375000,0.508197,0.400000,0.000000,0.500000,0.718750,0.362385,0.570423,0.569892,0.443709,...,1.000000,0.0,1.0,0.000000,0.513514,0.513514,0.333333,0.549889,1.000000,0.166667


In [8]:
# Build the label vector

# Sort to match user and video embeddings sort order
ratings = user_ratings.sort_values(['video_id', 'user_id'])
ratings = ratings.reset_index(drop=True)
ratings.head()

,user_id,video_id,like,watch_ratio
0,8,0,0,0.196883
1,14,0,1,1.417225
2,149,0,1,1.298425
3,206,0,1,1.669068
4,307,0,1,1.442694


In [9]:
# Round decimal of watch ratio so that the model does not learn too many
# decimals
ratings['watch_ratio'] = ratings['watch_ratio'].round(decimals=2)
ratings.head()

,user_id,video_id,like,watch_ratio
0,8,0,0,0.20
1,14,0,1,1.42
2,149,0,1,1.30
3,206,0,1,1.67
4,307,0,1,1.44


In [10]:
videos_content.shape, user_content.shape, ratings.shape

((10064065, 32), (10064065, 31), (10064065, 4))

For feeding the model, we need to scale independantly each group of vectors: `user`, `video` and `rating` vectors.

In [11]:
# Scale users
user_scaler = StandardScaler()
scaled_user_content = user_scaler.fit_transform(user_content)

# Scale videos
video_scaler = StandardScaler()
scaled_videos_content = video_scaler.fit_transform(videos_content)

# Scale watch ratios
rating_scaler = MinMaxScaler(feature_range=(-1, 1))
# Convert to numpy vector
scaled_ratings = rating_scaler.fit_transform(ratings['watch_ratio'].to_numpy().reshape(-1, 1))

## Train/validation split

For this model, we just split the dataset with the classic method. We just need to split independantly each family of vectors using the same random seed to keep the order in train and validation sets.

In [12]:
video_train, video_validation = train_test_split(
    scaled_videos_content, train_size=0.80, shuffle=True, random_state=MANUAL_RANDOM_SEED
)
user_train, user_validation = train_test_split(
    scaled_user_content, train_size=0.80, shuffle=True, random_state=MANUAL_RANDOM_SEED
)
y_train, y_validation = train_test_split(
    scaled_ratings, train_size=0.80, shuffle=True, random_state=MANUAL_RANDOM_SEED
)
ratings_train, ratings_validation = train_test_split(
    ratings, train_size=0.80, shuffle=True, random_state=MANUAL_RANDOM_SEED
) 

In [13]:
video_train.shape, user_train.shape, y_train.shape, ratings_train.shape

((8051252, 32), (8051252, 31), (8051252, 1), (8051252, 4))

## Build the network

As said before, we will have 2 separate networks: one for the user and the other one for the videos. These networks will then encode vectors so that we can apply a dot product afterward to approximate the watch ration for a give pair of user and video.

In [14]:
# Number of feature for encoding users and videos
num_outputs = 32
tf.random.set_seed(1)

# User neural network
user_NN = tf.keras.models.Sequential(
    [
        tf.keras.layers.Dense(
            256,
            activation="relu",
            name="user_layer_1",
        ),
        tf.keras.layers.Dense(
            128,
            activation="relu",
            name="user_layer_2",
        ),
        tf.keras.layers.Dense(
            num_outputs,
            activation="linear",
            name="user_layer_3",
        ),
    ], name="user_NN"
)

# Video neural network
video_NN = tf.keras.models.Sequential(
    [
        tf.keras.layers.Dense(
            256,
            activation="relu",
            name="video_layer_1",
        ),
        tf.keras.layers.Dense(
            128,
            activation="relu",
            name="video_layer_2",
        ),
        tf.keras.layers.Dense(
            num_outputs,
            activation="linear",
            name="video_layer_3",
        ),
    ], name="video_NN"
)

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(user_train.shape[1],), name="user_input")
vu = user_NN(input_user)
vu = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1), name='l2_user')(vu)

# create the item input and point to the base network
input_video = tf.keras.layers.Input(shape=(video_train.shape[1],), name="video_input")
vm = video_NN(input_video)
vm = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1), name='l2_video')(vm)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = tf.keras.Model([input_user, input_video], output)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 31)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ video_input         │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_NN             │ (None, 32)        │     45,216 │ user_input[0][0]  │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ video_NN            │ (None, 32)        │     45,472 │ video_input[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ l2_user (Lambda)    │ (None, 32)        │          0 │ user_NN[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ l2_video (Lambda)   │ (None, 32)        │          0 │ video_NN[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ l2_user[0][0],    │
│                     │                   │            │ l2_video[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 90,688 (354.25 KB)

 Trainable params: 90,688 (354.25 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# Define cost function, optimizer and compile the model
tf.random.set_seed(MANUAL_RANDOM_SEED)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=1e-1)
model.compile(optimizer=opt, loss=cost_fn)

## Train the model

In [16]:
tf.random.set_seed(MANUAL_RANDOM_SEED)
model.fit(
    [user_train, video_train],
    y_train,
    epochs=2,
    validation_data=([user_validation, video_validation], y_validation),
)

Epoch 1/2
251602/251602 ━━━━━━━━━━━━━━━━━━━━ 328s 1ms/step - loss: 0.0326 - val_loss: 0.0323
Epoch 2/2
251602/251602 ━━━━━━━━━━━━━━━━━━━━ 331s 1ms/step - loss: 0.0318 - val_loss: 0.0321


In [17]:
# Check the loss function over the validation set
model.evaluate([user_validation, video_validation], y_validation)

62901/62901 ━━━━━━━━━━━━━━━━━━━━ 27s 433us/step - loss: 0.0322


0.032148782163858414

In [18]:
user_validation.shape, video_validation.shape, y_validation.shape

((2012813, 31), (2012813, 32), (2012813, 1))

## Evaluate the predictions

We first make a little attempt to see how to recommend video for a specific user. Then I will do it for all and compute Precision@K, NDCG@K and ILD@K metrics to compare with the logistic regression we implemented before.

In [19]:
# Recommend videos with user 0
user_example = 0

# Get the video the user has interacted with
ratings_ex = ratings[ratings['user_id'] == user_example]
video_ex = videos_content.iloc[ratings_ex.index]
user_ex = user_content.iloc[ratings_ex.index]

# Apply normalisation
scaled_video_ex = video_scaler.transform(video_ex)
scaled_user_ex = user_scaler.transform(user_ex)

# Predict by passing the vectors through the network
y_pred = model.predict([scaled_user_ex, scaled_video_ex])

# Unscale the data
y_pred = rating_scaler.inverse_transform(y_pred)

# Get recommendations dataframe
recommendations_ex = pd.concat([ratings_ex.reset_index(), pd.DataFrame(y_pred, columns=['prediction'])], axis=1)
recommendations_ex
recommendations_ex.sort_values('prediction', ascending=False)

68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


,index,user_id,video_id,like,watch_ratio,prediction
854,4125567,0,4040,1,4.51,2.701261
94,652435,0,600,1,4.13,2.656316
1703,7966409,0,8298,1,4.14,2.471189
1718,8024405,0,8366,0,0.23,2.462410
51,255588,0,314,1,4.20,2.415539
...,...,...,...,...,...,...
1749,8220870,0,8537,0,0.10,0.090116
912,4355823,0,4194,0,0.10,0.089597
1253,5979755,0,5848,0,0.10,0.088324
261,1384986,0,1134,0,0.10,0.088037


In [20]:
recommendations: pd.DataFrame = None

i = 0
for user_id, user_ratings in ratings_validation.groupby('user_id'):
    videos = videos_content.iloc[user_ratings.index]
    user = user_content.iloc[user_ratings.index]

    scaled_user_video = video_scaler.transform(videos)
    scaled_user_data = user_scaler.transform(user)

    y_pred = model.predict([scaled_user_data, scaled_user_video], verbose=0)
    y_pred = rating_scaler.inverse_transform(y_pred)

    recommendations_for_user = pd.concat([user_ratings.reset_index(), pd.DataFrame(y_pred, columns=['prediction'])], axis=1)
    recommendations_for_user = recommendations_for_user.sort_values('prediction', ascending=False)

    if recommendations is None:
        recommendations = recommendations_for_user
    else:
        recommendations = pd.concat([recommendations, recommendations_for_user])
    i+=1
    if i % 20 == 0:
        print(f'Step {i}')

Step 20
Step 40
Step 60
Step 80
Step 100
Step 120
Step 140
Step 160
Step 180
Step 200
Step 220
Step 240
Step 260
Step 280
Step 300
Step 320
Step 340
Step 360
Step 380
Step 400
Step 420
Step 440
Step 460
Step 480
Step 500
Step 520
Step 540
Step 560
Step 580
Step 600
Step 620
Step 640
Step 660
Step 680
Step 700
Step 720
Step 740
Step 760
Step 780
Step 800
Step 820
Step 840
Step 860
Step 880
Step 900
Step 920
Step 940
Step 960
Step 980
Step 1000
Step 1020
Step 1040
Step 1060
Step 1080
Step 1100
Step 1120
Step 1140
Step 1160
Step 1180
Step 1200
Step 1220
Step 1240
Step 1260
Step 1280
Step 1300
Step 1320
Step 1340
Step 1360
Step 1380
Step 1400
Step 1420
Step 1440
Step 1460
Step 1480
Step 1500
Step 1520
Step 1540
Step 1560
Step 1580
Step 1600
Step 1620
Step 1640
Step 1660
Step 1680
Step 1700
Step 1720
Step 1740
Step 1760
Step 1780
Step 1800
Step 1820
Step 1840
Step 1860
Step 1880
Step 1900
Step 1920
Step 1940
Step 1960
Step 1980
Step 2000
Step 2020
Step 2040
Step 2060
Step 2080
Step 2100
Ste

In [ ]:
# Select columns in the recommendation dataframe to compute the diversity metric on
if 'index' in recommendations.columns:
    recommendations = recommendations.drop('index', axis=1)
recommendations = recommendations.merge(videos_tags, on='video_id')
recommendations = recommendations.merge(videos_content_full.iloc[:, 2:], on='video_id')

recommendations.head()

,user_id,video_id,like,watch_ratio,prediction,tag_0,tag_1,tag_2,tag_3,tag_4,...,tag_28,tag_29,tag_30,show_cnt,play_cnt,valid_play_cnt,like_cnt,comment_cnt,share_cnt,video_duration
0,0,8298,1,4.14,2.471189,0,0,0,0,0,...,0,0,0,580837,453961,352644,7593,112,28,3270
1,0,8366,0,0.23,2.462410,0,0,0,0,0,...,0,0,0,531519,520237,414062,2856,35,4,3300
2,0,9365,1,3.22,2.096799,0,0,0,0,0,...,0,0,0,3151,3185,2059,198,0,0,4021
3,0,1305,0,0.24,1.976816,0,0,0,0,0,...,0,0,0,217373,224249,182468,1415,3,1,3390
4,0,1709,1,3.39,1.955445,0,0,0,0,0,...,1,0,0,215369,221825,153069,1553,65,4,3994


In [36]:
# Select columns in the recommendation dataframe to compute the diversity metric on
video_content_columns = recommendations.columns[5:]
video_content_columns

Index(['tag_0', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7',
       'tag_8', 'tag_9', 'tag_10', 'tag_11', 'tag_12', 'tag_13', 'tag_14',
       'tag_15', 'tag_16', 'tag_17', 'tag_18', 'tag_19', 'tag_20', 'tag_21',
       'tag_22', 'tag_23', 'tag_24', 'tag_25', 'tag_26', 'tag_27', 'tag_28',
       'tag_29', 'tag_30', 'show_cnt', 'play_cnt', 'valid_play_cnt',
       'like_cnt', 'comment_cnt', 'share_cnt', 'video_duration'],
      dtype='object')

In [37]:
# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)
    diversity = metrics.inter_list_diversity_at_k(recommendations, video_content_columns, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print(f'ILD@{k} -> {diversity}')
    print()

Precision@5 -> 0.7861204013377926
NDCG@5 -> 0.9071451484301563
ILD@5 -> 0.7853266706439007

Precision@10 -> 0.7693004459308808
NDCG@10 -> 0.90585546083173
ILD@10 -> 0.7991469707233336

Precision@15 -> 0.7547658862876255
NDCG@15 -> 0.9046604039682422
ILD@15 -> 0.8082633315688594

Precision@20 -> 0.7427048494983277
NDCG@20 -> 0.9042469544598459
ILD@20 -> 0.8177877831444081



## Interpretations

The metrics above show the neural network works well with a Precision@5 at 78%. However, it is outperformed by a simple logistic regression. Plus, logistic regression is extremly simple to use compare to training a whole neural network. I think it needs way more training epochs with the very generalized features I use here. Indeed, I did not want to use features such as the author or the music as almost all author only published 1 video and that leads to overfit on prioritizing the author id in the learning process. That is why I will stick with a simple logistic regression to build the final recommender system.